# Creative OS — Node Engine Prototype

Implements the Hexagonal Blueprint node transition logic from `spec/hexagonal_blueprint_v05.md`.

**Rules enforced:**
- Simultaneous activation limit: max 3 node **types** active at once; exceeding without prior Z fires Lore Creep
- Floor nodes (π, project-elevated H) do not consume a slot
- A tracks per-entity; slot count tracks node type (multiple A instances = 1 A slot)
- **Slot count ≠ state list length.** State line lists all active instances; slot count counts active types.
- Z fires in three modes: Complete, Truncated, Partial/External. Complete may be terminal (entity exits story).
- S deactivation: forcing function and merge-completion are separate events (Ruling 6.3)
- D auto-activates when S deactivates with a shared goal still active (Ruling 6.6 — formalized)

In [ ]:
from typing import Optional


Z_MODES = {'Complete', 'Truncated', 'Partial/External'}


class NodeEngine:
    """Tracks node state against Hexagonal Blueprint rules (v0.5)."""

    def __init__(self, project: str, floor_nodes: Optional[set] = None):
        self.project = project
        self.floor_nodes = floor_nodes or set()
        self.active: dict[str, list[str]] = {}
        self.z_mode: Optional[str] = None
        self.deferred_z: list[dict] = []
        self.violations: list[dict] = []
        self.physics_violations: list[dict] = []
        self.declared_state: dict[str, dict] = {}
        self._log: list[dict] = []

    def _remind_declared_state(self, name: str) -> None:
        d = self.declared_state.get(name)
        if d:
            print(f'  ℹ  {name} last declared at {d["scene"]}: {d["description"]}')

    def slot_count(self) -> int:
        return sum(1 for n in self.active if n not in self.floor_nodes)

    def _active_display(self) -> list[str]:
        out = []
        for node, entities in self.active.items():
            tag = '[floor]' if node in self.floor_nodes else ''
            if node == 'A':
                out.extend(f'A({e})' for e in entities)
            else:
                out.append(f'{node}{tag}')
        return out

    def activate(self, node: str, scene: str, entity: str = '_',
                 z_mode: Optional[str] = None) -> None:
        if node != 'Z' and node not in self.floor_nodes:
            if self.slot_count() >= 3:
                v = {'type': 'LORE_CREEP', 'scene': scene, 'node': node,
                     'slot_count': self.slot_count(), 'active': self._active_display()}
                self.violations.append(v)
                print(f'  ⚠  LORE_CREEP at {scene}: {node} would make '
                      f'slot_count={self.slot_count() + 1} without prior Z')

        if node not in self.active:
            self.active[node] = []
        if entity not in self.active[node]:
            self.active[node].append(entity)

        # Surface any declared state for entities appearing in this activation --
        # a reminder, not a judgment.
        for name in entity.split('+'):
            self._remind_declared_state(name)

        if node == 'Z':
            self.z_mode = z_mode or 'Complete'
            if self.z_mode in ('Truncated', 'Partial/External'):
                self.deferred_z.append({'entity': entity, 'mode': self.z_mode,
                                        'opened_at': scene})
        self._record(scene, 'ACTIVATE', node, entity, z_mode=z_mode)

    def declare_state(self, entity: str, description: str, scene: str) -> None:
        """Record an entity's current core conviction/thesis -- typically called

        right after a Z resolves, or at any other beat that settles what a
        character now believes and why. Not a judgment call itself; it's the
        thing a later flag_physics_violation() call gets checked against, and
        what activate() surfaces automatically the next time this entity
        reappears -- so the trace-writer doesn't have to recall or re-search
        prior chapters/traces from memory.
        """
        self.declared_state[entity] = {'description': description, 'scene': scene}

    def deactivate(self, node: str, scene: str, entity: str = '_',
                   forcing_function: Optional[str] = None,
                   goal_active: bool = False) -> None:
        if node == 'Z':
            self.z_mode = None
            stale = [d for d in self.deferred_z if d['entity'] == entity]
            if stale:
                print(f'  ⚠  Z deactivated for {entity} at {scene}, but {len(stale)} '
                      f'deferred entry(ies) for {entity} remain unresolved: '
                      f'{[(d["mode"], d["opened_at"]) for d in stale]}. '
                      f'deactivate() does not clear deferrals -- if this Z was meant '
                      f'to resolve them, call resolve_deferred_z() instead, or '
                      f'reactivate as z_mode=\'Complete\' if it never should have '
                      f'been deferred in the first place.')

        if node in self.active:
            if entity in self.active[node]:
                self.active[node].remove(entity)
            if not self.active[node]:
                del self.active[node]

        self._record(scene, 'DEACTIVATE', node, entity,
                     forcing_function=forcing_function)

        # Ruling 6.6 (formalized): D auto-activates when S deactivates with goal active
        if node == 'S' and goal_active:
            ff = f' [S forcing function: {forcing_function}]' if forcing_function else ''
            print(f'  ▷  D (Unified Pursuit) auto-activated at {scene}{ff}')
            self.activate('D', scene)

    def resolve_deferred_z(self, scene: str, entity: str = '_',
                           in_scene: bool = False) -> None:
        """Resolve a deferred Z for the given entity.

        in_scene=True: only closes Partial/External entries (completed without
        deferral — fires and resolves in the same scene block). Truncated entries
        for the same entity remain open and continue watching.

        in_scene=False (default): closes all deferred entries for the entity
        (full deferred resolution — Ruling 6.4 conditions met).
        """
        if in_scene:
            match = lambda d: d['entity'] == entity and d['mode'] == 'Partial/External'
        else:
            match = lambda d: d['entity'] == entity

        still_open = [d for d in self.deferred_z if not match(d)]
        resolved   = [d for d in self.deferred_z if match(d)]
        self.deferred_z = still_open

        for r in resolved:
            if in_scene:
                print(f'  ✓  Z PARTIAL/EXTERNAL → COMPLETE (in-scene) at {scene}: '
                      f"{r['entity']}")
            else:
                print(f'  ✓  DEFERRED Z RESOLVED at {scene}: '
                      f"{r['entity']} ({r['mode']} from {r['opened_at']}) → Complete")
        self._record(scene, 'Z_RESOLVED', 'Z', entity)

    def flag_physics_violation(self, scene: str, entity: str, description: str,
                                contradicts: Optional[str] = None) -> None:
        """Log a PHYSICS_VIOLATION per Ruling 6.7's test: does the character's core

        thesis/conviction change without being earned on the page? Unlike Lore Creep
        (auto-detected from the 3-slot limit), this is a judgment call the trace-writer
        makes and records explicitly -- the engine has no way to infer character
        psychology on its own, only to formally track a call once it's been made.

        `contradicts` should name the prior established state this entity's current
        behavior fails to earn a transition away from. If omitted, it's pulled from
        declared_state automatically -- pass it explicitly only when the contradicted
        state was never formally declared (e.g. it lives only in prior trace prose).
        """
        if contradicts is None:
            d = self.declared_state.get(entity)
            contradicts = (f'{d["scene"]}: {d["description"]}' if d
                            else '(no declared_state on file for this entity -- '
                                 'contradicts undocumented)')
        v = {'type': 'PHYSICS_VIOLATION', 'scene': scene, 'entity': entity,
             'description': description, 'contradicts': contradicts}
        self.physics_violations.append(v)
        print(f'  ✘  PHYSICS_VIOLATION at {scene} ({entity}): {description}')
        print(f'      contradicts: {contradicts}')

    def report(self, scene: str) -> None:
        # Heuristic: if a declared entity's name shows up in the scene label
        # itself (case-insensitive), surface the reminder even though nothing
        # is being activated -- this is exactly the gap that let the Ch.8-alt
        # Thorne reappearance pass without a reminder the first time around:
        # his return there is a plain report(), not an activate().
        scene_lower = scene.lower()
        for name in self.declared_state:
            if name.lower() in scene_lower:
                self._remind_declared_state(name)

        z_info = f' | Z:{self.z_mode}' if self.z_mode else ''
        print(f'  {scene:<50} active={self._active_display()}  '
              f'slots={self.slot_count()}{z_info}')

    def _record(self, scene, action, node, entity, z_mode=None, forcing_function=None):
        self._log.append({
            'scene': scene, 'action': action, 'node': node, 'entity': entity,
            'z_mode': z_mode, 'forcing_function': forcing_function,
            'slot_count': self.slot_count(), 'active': self._active_display(),
        })

    def summary(self) -> None:
        print(f'\n── {self.project} Summary ──')
        print(f'  Violations (Lore Creep):      {len(self.violations)}')
        print(f'  Violations (Physics):         {len(self.physics_violations)}')
        if self.physics_violations:
            for p in self.physics_violations:
                print(f'    → {p["entity"]} at {p["scene"]}: {p["description"]}')
        print(f'  Deferred Z still open:         {len(self.deferred_z)}')
        if self.deferred_z:
            for d in self.deferred_z:
                print(f'    → {d["entity"]} ({d["mode"]} opened at {d["opened_at"]})')
        print(f'  D (Unified Pursuit) active:    {"D" in self.active}')
        print(f'  Active nodes at close:         {self._active_display()}')
        print(f'  Slot count at close:           {self.slot_count()}')
        print(f'  Declared states on file:       {list(self.declared_state.keys())}')

## Veritas — Setup

H is elevated to floor status per Ruling 6.1 (the Hum never recedes — premise permanently dissolves structural bounds).

In [2]:
# Veritas project: H on floor
eng = NodeEngine(project='Veritas', floor_nodes={'H'})

print('Project:', eng.project)
print('Floor nodes:', eng.floor_nodes)
print('π always present (not tracked as slot — structural floor)')

Project: Veritas
Floor nodes: {'H'}
π always present (not tracked as slot — structural floor)


## Trace 001 — Prologue + Chapter 1

In [3]:
print('── Prologue ──')
eng.activate('W', 'prologue.open')
eng.report('prologue.open')

# Hum arrives — H activates (elevates to floor), W deactivates
eng.activate('H', 'prologue.hum_arrives')  # no slot consumed — floor node
eng.deactivate('W', 'prologue.hum_arrives')  # active tension now detected
eng.report('prologue.hum_arrives')
# γ fires (generative — age of Veritas begins)

print('\n── Chapter 1: Cassie thread ──')
# Dana kitchen scene — A activates (Truth vs. Lie in balance)
eng.activate('A', 'ch1.kitchen_dana', entity='Cassie_domestic')
eng.report('ch1.kitchen_dana')  # slots: A=1

# Car ride — A holds, σ elevated on path
eng.report('ch1.car_ride')

# Boardroom — Z fires (Complete) before slot limit reached
eng.activate('Z', 'ch1.boardroom', z_mode='Complete')  # A + Z = 2 slots
eng.report('ch1.boardroom')
# α, δ, τ fire (duration-dependent — boardroom required full scene)

eng.deactivate('Z', 'ch1.boardroom_close')  # Z deactivates, narrative back in motion
eng.report('ch1.boardroom_close')

print('\n── Chapter 1: Valeria thread ──')
# γ fires (new thread — Valeria introduced)
# A(Valeria) activates — truth vs. necessary lie
eng.activate('A', 'ch1.precinct_intro', entity='Valeria')
eng.report('ch1.precinct_intro')  # A(Cassie_domestic) + A(Valeria) = 1 slot

# Reyes confession — A(precinct) deactivates, A(Valeria) persists
eng.deactivate('A', 'ch1.reyes_confession', entity='Cassie_domestic')
# Note: Cassie_domestic was the stand-in for the precinct's institutional A
# In this trace, precinct A was bundled with Cassie thread — refine in v2
eng.report('ch1.reyes_confession')

# Chapter 1 close — α fires toward Ch.2
eng.report('ch1.close')
print('\nTrace 001 complete.')

── Prologue ──
  prologue.open                                      active=['W']  slots=1
  prologue.hum_arrives                               active=['H[floor]']  slots=0

── Chapter 1: Cassie thread ──
  ch1.kitchen_dana                                   active=['H[floor]', 'A(Cassie_domestic)']  slots=1
  ch1.car_ride                                       active=['H[floor]', 'A(Cassie_domestic)']  slots=1
  ch1.boardroom                                      active=['H[floor]', 'A(Cassie_domestic)', 'Z']  slots=2 | Z:Complete
  ch1.boardroom_close                                active=['H[floor]', 'A(Cassie_domestic)']  slots=1

── Chapter 1: Valeria thread ──
  ch1.precinct_intro                                 active=['H[floor]', 'A(Cassie_domestic)', 'A(Valeria)']  slots=1
  ch1.reyes_confession                               active=['H[floor]', 'A(Valeria)']  slots=1
  ch1.close                                          active=['H[floor]', 'A(Valeria)']  slots=1

Trace 001 complete

## Trace 002 — Chapter 2

In [4]:
print('── Chapter 2: Cassie thread ──')

# Apartment — A(Cassie+Dana) activates, Z fires (Complete — personal coherence check)
eng.activate('A', 'ch2.apartment_dana', entity='Cassie+Dana')
eng.activate('Z', 'ch2.apartment_z', z_mode='Complete')  # A(Valeria)+A(Cassie+Dana)+Z
eng.report('ch2.apartment_z')  # slots: A=1 (two instances), Z=1 → total 2

# Confession complete — vase shatters, A(Cassie+Dana) resolves, Z deactivates
eng.deactivate('A', 'ch2.vase_shatters', entity='Cassie+Dana')
eng.deactivate('Z', 'ch2.vase_shatters')
eng.report('ch2.vase_shatters')  # slots: A(Valeria)=1

# Office — no Z (no false self remaining to confirm)
# α blocked (spin attempt fails), δ fires when she runs
eng.report('ch2.office_collapse')

print('\n── Chapter 2: Valeria thread ──')

# Precinct — Vance confession
# α, δ fire (cash hits floor, precinct state permanently shifted)
eng.report('ch2.precinct_vance')

# Chapter 2 close — α fires for S (convergence initiated by Valeria reading the text)
eng.activate('S', 'ch2.close_text_received')  # S in transition (α fired)
eng.report('ch2.close_text_received')  # slots: A(Valeria)=1, S=1 → total 2

print('\nTrace 002 complete.')

── Chapter 2: Cassie thread ──
  ch2.apartment_z                                    active=['H[floor]', 'A(Valeria)', 'A(Cassie+Dana)', 'Z']  slots=2 | Z:Complete
  ch2.vase_shatters                                  active=['H[floor]', 'A(Valeria)']  slots=1
  ch2.office_collapse                                active=['H[floor]', 'A(Valeria)']  slots=1

── Chapter 2: Valeria thread ──
  ch2.precinct_vance                                 active=['H[floor]', 'A(Valeria)']  slots=1
  ch2.close_text_received                            active=['H[floor]', 'A(Valeria)', 'S']  slots=2

Trace 002 complete.


## Trace 003 — Chapter 3

In [5]:
print('── Chapter 3: Drive → convergence ──')

# Drive — S in transition, σ at peak, τ fires (duration-dependent approach sequence)
eng.report('ch3.drive_truth_mob')  # slots: A(Valeria)=1, S=1 → 2

# Cassie enters bank — S fully activates (δ fires for S)
eng.report('ch3.cassie_enters_bank')  # slots: A(Valeria)=1, S=1 → 2

# Confrontation — Z fires (Truncated: mob will interrupt before it completes)
# entity='Valeria' so the deferred Z watch list names the right character
eng.activate('Z', 'ch3.confrontation_gun_drawn', entity='Valeria', z_mode='Truncated')
eng.report('ch3.confrontation_gun_drawn')  # A + S + Z = 3 — AT LIMIT

# Mob invades — Z truncated (cut off), S deactivates (merge complete)
# Ruling 6.3: mob is forcing function; merge-completion is the cause
eng.deactivate('Z', 'ch3.mob_invades', entity='Valeria')  # deferred Z watch opened for Valeria
eng.deactivate('S', 'ch3.mob_invades',
               forcing_function='Purified mob',
               goal_active=True)  # goal (Shadow Ledger) still active → null-gap logged
eng.report('ch3.mob_invades')  # slots: A(Valeria)=1

# Smoke grenade — δ fires (Valeria's purpose shifts: arrest → collaborate)
eng.report('ch3.smoke_grenade')

# Tunnel — A(Cassie+Valeria) activates (TILTED — Valeria's armor intact)
# Cassie calls out Valeria on the car. Valeria gets the last word.
eng.activate('A', 'ch3.tunnel_exchange', entity='Cassie+Valeria_tilted')
eng.report('ch3.tunnel_exchange')  # A(Valeria) + A(Cassie+Valeria) = 1 slot

# Chapter close — α fires toward Ch.4, darkness
eng.report('ch3.close_darkness')

print('\nTrace 003 complete.')

── Chapter 3: Drive → convergence ──
  ch3.drive_truth_mob                                active=['H[floor]', 'A(Valeria)', 'S']  slots=2
  ch3.cassie_enters_bank                             active=['H[floor]', 'A(Valeria)', 'S']  slots=2
  ch3.confrontation_gun_drawn                        active=['H[floor]', 'A(Valeria)', 'S', 'Z']  slots=3 | Z:Truncated
  ▷  D (Unified Pursuit) auto-activated at ch3.mob_invades [S forcing function: Purified mob]
  ch3.mob_invades                                    active=['H[floor]', 'A(Valeria)', 'D']  slots=2
  ch3.smoke_grenade                                  active=['H[floor]', 'A(Valeria)', 'D']  slots=2
  ch3.tunnel_exchange                                active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch3.close_darkness                                 active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2

Trace 003 complete.


## Summary

eng.summary()

print('\n── Deferred Z chain (Veritas) ──')
print('  Ch.1 boardroom:     Complete')
print('  Ch.2 apartment:     Complete')
print('  Ch.3 bank:          Truncated → watching Valeria → resolves Ch.8')
print('  Ch.5 trailer:       Partial/External → Complete (in-scene)')
print('  Ch.8 (pending):     Complete/self-confessed → resolves Ch.3 Truncated')

print('\n── D (Unified Pursuit) ──')
print('  Activated:  ch3.mob_invades (auto — S completed, Ledger+Thorne goal active)')
print('  Still open: pursuing Thorne + stopping Phase 5 through Ch.5 close')
print('  Deactivates: at G (full resolution) or F (resolution approaching)')

print('\n── 3-node limit ──')
peak = max((e['slot_count'] for e in eng._log), default=0)
print(f'  Peak slot count across all traces: {peak}')
print(f'  Lore Creep violations: {len(eng.violations)}')
if not eng.violations:
    print('  ✓ Limit respected across Ch.1–5')

In [6]:
print('── Chapter 4: Silence of the Lambs ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print('Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)')
print()

# Tunnel — Cassie confesses PSH role ("Predictive Social Harmonization")
# α fires: Valeria's understanding of what the Hum is begins to shift
# σ elevated: path toward Thorne now named and specific
eng.report('ch4.tunnel_psh_confession')

# City Hall Station — catatonic victims, Operative 4 with HUD
# σ peak: sealed space, actively hunted
# τ fires: chandelier approach sequence (duration-dependent — full arc required)
# 3rd τ instance (Ch.1 boardroom, Ch.3 drive, Ch.4 chandelier — all approach sequences)
eng.report('ch4.station_silent_gallery')
eng.report('ch4.cleaners_emerge')
eng.report('ch4.chandelier_escape')  # τ

# Foreman's trailer — Shadow Ledger biometric auth
# Note: acoustic maps and St. Jude's logs are ALSO staged here in the final manuscript
# but per Defect 5.1 those beats appear as-written in Ch.5 (stitch artifact)
# Tracing as-written: only Thorne.pdf fires in Ch.4
eng.report('ch4.shadow_ledger_open')

# Director_E_Thorne.pdf — Z fires (Complete, entity=Valeria)
# Identity collapse: Thorne mentored her father, shaped her career, ran the trials.
# "she'd been preparing herself to enforce his experiment" (line 746)
# Miller's "Val! We have to go!" arrives at line 747 — AFTER completion. Not Truncated.
# This is institutional-identity collapse — distinct from the Ch.3 moral-act Z (Truncated).
# Ch.3 deferred Z remains open and independent.
eng.activate('Z', 'ch4.thorne_identity_collapse', entity='Valeria', z_mode='Complete')
eng.report('ch4.thorne_identity_collapse')  # A + D + Z = 3 — AT LIMIT

# Z completes in-scene — identity processed, decision made: pursue Thorne
eng.deactivate('Z', 'ch4.z_complete_pursue', entity='Valeria')
eng.report('ch4.z_complete_pursue')  # slots return to 2

# Iron door breached — flee (breaching charge, Cleaners, laser sights)
# Chapter closes at a dead sprint — σ
eng.report('ch4.door_breached_flee')

print()
print('Trace 004 complete.')
print('Z Complete (entity=Valeria) fired and resolved in-scene at Thorne identity collapse.')
print('Peak slot at Z: 3 (A + D + Z). Limit respected.')
print(f'τ: 3rd instance (chandelier approach). All approach sequences so far.')
print('Slot count at close: 2.')
print('Deferred Z: still open — Valeria, Truncated, ch3.confrontation_gun_drawn.')

── Chapter 4: Silence of the Lambs ──
Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2
Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)

  ch4.tunnel_psh_confession                          active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch4.station_silent_gallery                         active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch4.cleaners_emerge                                active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch4.chandelier_escape                              active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch4.shadow_ledger_open                             active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch4.thorne_identity_collapse                       active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D', 'Z']  slots=3 | Z:Complete
  ch4.z_complete_pursue                  

## Trace 005 — Chapter 5: The Shattered Shield

In [7]:
print('── Chapter 5: The Shattered Shield ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print('Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)')
print()
print('MANUSCRIPT DEFECTS LOGGED:')
print('  Defect 5.1 — Scene duplication: Trailer/Toughbook/Shadow-Ledger staged twice.')
print('    Acoustic maps + St. Jude\'s logs appear as-written in Ch.5; belong in Ch.4 post-fix.')
print('    Tracing as-written. Affected scenes annotated below.')
print('  Defect 5.2 — Kinetic break: Ch.4 closes at dead sprint; Ch.5 opens stationary.')
print('    WANDER_ALARM_KINETIC_DEVIATION-class. Framework catch confirmed.')
print()

# Construction site exterior — Miller processing guilt, Cassie + Valeria regroup
# σ active. D persists. Ch.5 proper begins here.
eng.report('ch5.construction_site_exterior')

# [Defect 5.1 — as-written] Foreman's trailer re-staged
# Shadow Ledger opened again. Acoustic maps (Manhattan-Zero.pdf) revealed.
# γ fires: city-as-instrument revelation — new structural information enters system
# Post-fix: this γ belongs in the Ch.4 Toughbook session
eng.report('ch5.acoustic_maps_reveal')  # γ [Defect 5.1]

# [Defect 5.1 — as-written] St. Jude's medical logs
# δ fires: full scope of what Cassie enabled lands
# NOT Z for Cassie — scope expansion, not coherence checkpoint
# Cassie's coherence was confirmed in Ch.3 (bank). This is δ + γ, not Z.
# Post-fix: this δ belongs in the Ch.4 Toughbook session
eng.report('ch5.st_judes_logs')  # δ [Defect 5.1]

# Helicopter arrives — flashbangs, windows shattered
# σ spikes to peak — external threat forces exit
eng.report('ch5.helicopter_flashbangs')

# Crane climb
# τ fires: duration-dependent approach sequence (full climb arc required)
# 4th τ instance (Ch.1, Ch.3, Ch.4, Ch.5 — all approach sequences)
# WATCH: τ at 4/4 approach sequences. Hunt non-approach instance in Ch.6–8 before ruling.
eng.report('ch5.crane_climb')  # τ (4th instance — all approaches)

# Miller — Sonic Projector fires. Hum removed. Clarity arrives.
# Z fires: Complete, TERMINAL
# "he'd wanted to be one of the good guys, and somewhere along the way
#  he'd stopped being good without even noticing" — identity collapse, lines 950–951
# Miller steps deliberately (lines 957–961) — "the choice"
# Valeria's lunge arrives AFTER completion — not Truncated.
# FIRST TERMINAL Z IN CORPUS — entity's death is the resolution.
# Flag opened. Sub-mode ruling held until two more instances.
# D-COMPOSITION EVENT: Miller exits merged unit.
# D persists — goal intact, Valeria+Cassie carry the pursuit forward.
eng.activate('Z', 'ch5.crane_miller_clarity', entity='Miller', z_mode='Complete')
eng.report('ch5.crane_miller_clarity')  # A + D + Z = 3 — AT LIMIT (terminal Complete)

eng.deactivate('Z', 'ch5.crane_miller_falls', entity='Miller')
eng.report('ch5.crane_miller_falls')  # δ (terminal) — slots return to 2

# Truth Bomb detonation — Shadow Ledger broadcast fires
# γ fires: new state created, truth goes global, age of secrets detonated
# No new nodes: story still in motion toward Thorne and Phase 5
# F does not fire: multiple threads still unresolved
eng.report('ch5.truth_bomb')  # γ

# Chapter close — "the truth didn't just hurt. It cleansed."
# δ fires: world-state has shifted
# Z does NOT fire: thematic statement, not character-level identity-collapse
# Z VARIANT WATCH: 2nd instance of thematic chapter-close beat logged as δ
#   (Ch.3 tunnel close + Ch.5 here). One more = ruling threshold met.
# D persists: shared goal (Thorne + stopping Phase 5) still active
eng.report('ch5.close_truth_cleanses')

print()
print('Trace 005 complete.')
print('Defects 5.1 + 5.2 logged. Traced as-written.')
print('Miller Z: Complete, TERMINAL — first terminal Z in corpus. Flag opened.')
print('D-composition event: Miller exits unit. D persists with Valeria+Cassie.')
print('τ: 4/4 approach sequences. Hunt non-approach in Ch.6–8 before ruling.')
print('Z variant watch: 2/3 instances. One more = ruling.')
print('Ch.3 Truncated Z still open — watching Valeria, resolves Ch.8.')

── Chapter 5: The Shattered Shield ──
Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2
Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)

MANUSCRIPT DEFECTS LOGGED:
  Defect 5.1 — Scene duplication: Trailer/Toughbook/Shadow-Ledger staged twice.
    Acoustic maps + St. Jude's logs appear as-written in Ch.5; belong in Ch.4 post-fix.
    Tracing as-written. Affected scenes annotated below.
  Defect 5.2 — Kinetic break: Ch.4 closes at dead sprint; Ch.5 opens stationary.
    WANDER_ALARM_KINETIC_DEVIATION-class. Framework catch confirmed.

  ch5.construction_site_exterior                     active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch5.acoustic_maps_reveal                           active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch5.st_judes_logs                                  active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch5.helicopter_flashbangs                

## Trace 006 — Chapter 6: The Collapse

In [8]:
print('── Chapter 6: The Collapse ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print('POV: Thorne (Aegis Suite, 80th floor, Federal Building)')
print()
print('RULING 6.7 APPLIES: Hum-compelled disclosure is physics-consistent.')
print('Physics checks conviction not concealment. See spec Ruling 6.7.')
print()

# Aegis Suite — Thorne watches the city burn. Resonance map active.
# δ: POV shift — no node event. D persists (merged unit Valeria+Cassie still in motion).
# A(Valeria+Thorne) does NOT activate — phone call is α, not A (co-location not met).
eng.report('ch6.aegis_suite_opens')

# Thomas Knox's badge on the desk
# δ: weight of guilt named. σ: pressure of past action present.
eng.report('ch6.knox_badge')

# Shadow Ledger broadcast confirmed spreading (peer-to-peer, uncontainable)
# δ: world state shifts — files already loose.
eng.report('ch6.ledger_broadcast_spreading')

# Chen refuses Omega-Seven order — first refusal
# δ: command structure fractures from within.
# Ruling 6.7: Chen's confession on removing headset = forced disclosure, not drift.
eng.report('ch6.chen_refuses')

# Technician cascade — headsets off one by one, confessions follow
# Sustained α — multiple beats, no clean δ landing until Thorne's decision.
# SECOND instance of sustained α in traces (Ch.4 City Hall Station was first).
# WATCH: third instance = ruling threshold on sustained α as distinct marker.
eng.report('ch6.technician_cascade')

# St. Jude's memory — Marcus Reeves, 11 years old, behind two-way mirror
# σ: pressure of past action. NOT Z — flashback, not in-scene identity-collapse.
# Ruling 6.7: disclosure of cost (what trials did to Marcus Reeves) ≠ change of conviction.
# Thorne's conviction intact throughout: "I knew it would hurt them. I did it anyway."
eng.report('ch6.st_judes_memory')

# Valeria's call — breached secure channel, audio only
# α fires toward A(Valeria+Thorne) — initiation, not activation.
# A(Valeria+Thorne) deferred to Ch.8: physical co-location required for staged tension.
eng.report('ch6.valeria_call')

# Thorne evacuates facility, stays alone
# δ: operational apparatus dissolves.
eng.report('ch6.facility_evacuated')

# Thorne steps into hallway — "as a man with a confession"
# δ: commander → confessor state shift.
# NOT τ: Ruling 6.7 corollary — Hum-compelled transitions are forced disclosure events,
#   not earned transitions. τ requires character agency; this is compelled.
# NOT PHYSICS_VIOLATION: conviction intact throughout (Ruling 6.7).
eng.report('ch6.thorne_into_hallway')

print()
print('Trace 006 complete.')
print('D persists across POV shift — confirmed.')
print('A(Valeria+Thorne) deferred to Ch.8. Phone call = α, not A.')
print('No Z, no F, no new nodes. Slot count: 2 throughout.')
print('Ruling 6.7: Thorne confession = forced disclosure, physics-consistent.')
print('Sustained α: 2nd instance (Ch.4 City Hall + Ch.6 cascade). One more = ruling.')
print('τ: 4/4 approach sequences — Hum-compelled shift disqualified (Ruling 6.7 corollary).')

── Chapter 6: The Collapse ──
Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2
POV: Thorne (Aegis Suite, 80th floor, Federal Building)

RULING 6.7 APPLIES: Hum-compelled disclosure is physics-consistent.
Physics checks conviction not concealment. See spec Ruling 6.7.

  ch6.aegis_suite_opens                              active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch6.knox_badge                                     active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch6.ledger_broadcast_spreading                     active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch6.chen_refuses                                   active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch6.technician_cascade                             active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch6.st_judes_memory                                active=['H[floor]', 

## Summary — Ch.1 through Ch.6

In [9]:
eng.summary()

print('\n── Deferred Z chain (Veritas) ──')
print('  Ch.1 boardroom:   Complete (Cassie)')
print('  Ch.2 apartment:   Complete (Cassie)')
print('  Ch.3 bank:        Truncated → watching Valeria → resolves Ch.8')
print('  Ch.4 trailer:     Complete (Valeria — Thorne identity collapse)')
print('  Ch.5 crane:       Complete, TERMINAL (Miller — first terminal Z in corpus)')
print('  Ch.6:             No Z — Thorne in motion, not in identity-collapse')
print('  Ch.8 (pending):   Complete/self-confessed → resolves Ch.3 Truncated')

print('\n── D (Unified Pursuit) ──')
print('  Auto-activated:  ch3.mob_invades (S completed, Ledger+Thorne goal active)')
print('  Composition:     Valeria+Cassie (Miller exited at ch5.crane — D-composition event #1)')
print('  Still running:   through Ch.6 — goal not in sight of resolution')
print('  Deactivates at:  G (full resolution) or F (resolution approaching)')

print('\n── 3-node limit ──')
peak = max((e['slot_count'] for e in eng._log), default=0)
print(f'  Peak slot count Ch.1–6: {peak}')
print(f'  Hit at: Ch.3 bank | Ch.4 trailer (Valeria Z) | Ch.5 crane (Miller Z terminal)')
print(f'  Lore Creep violations:  {len(eng.violations)}')
if not eng.violations:
    print('  ✓ Limit respected throughout')

print('\n── Slot accounting ──')
print('  State line = all active node instances')
print('  Slot count = active node TYPES  (A = 1 slot regardless of instance count)')
print('  H[floor] = 0 slots. 3-node limit applies to slot count, not state list length.')

print('\n── Open watches ──')
print('  τ: 4/4 approach sequences. Hunt non-approach in Ch.7–13 before ruling.')
print('     Hum-compelled transitions disqualified (Ruling 6.7 corollary).')
print('  Z variant (thematic chapter-close): 2/3 instances. One more = ruling.')
print('  Sustained α: 2/3 instances (Ch.4 City Hall, Ch.6 cascade). One more = ruling.')
print('  Terminal Z: 1 instance (Miller). Two more needed before sub-mode ruling.')
print('  A(Valeria+Thorne): α fired at Ch.6 call. Activation deferred to Ch.8.')
print('  Manuscript: Defect 5.1 (duplication) + 5.2 (kinetic break) at Ch.4→Ch.5 seam.')


── Veritas Summary ──
  Violations (Lore Creep):      0
  Deferred Z still open:         1
    → Valeria (Truncated opened at ch3.confrontation_gun_drawn)
  D (Unified Pursuit) active:    True
  Active nodes at close:         ['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']
  Slot count at close:           2

── Deferred Z chain (Veritas) ──
  Ch.1 boardroom:   Complete (Cassie)
  Ch.2 apartment:   Complete (Cassie)
  Ch.3 bank:        Truncated → watching Valeria → resolves Ch.8
  Ch.4 trailer:     Complete (Valeria — Thorne identity collapse)
  Ch.5 crane:       Complete, TERMINAL (Miller — first terminal Z in corpus)
  Ch.6:             No Z — Thorne in motion, not in identity-collapse
  Ch.8 (pending):   Complete/self-confessed → resolves Ch.3 Truncated

── D (Unified Pursuit) ──
  Auto-activated:  ch3.mob_invades (S completed, Ledger+Thorne goal active)
  Composition:     Valeria+Cassie (Miller exited at ch5.crane — D-composition event #1)
  Still running:   through Ch

## Trace 007 — Chapter 7: Power, Parse, Path

In [9]:
print('── Chapter 7: Power, Parse, Path ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print('Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)')
print()

# Service tunnel — Miller grief, Hum pressure muffled but present
# σ: pressure/conflict path. D persists (unified pursuit continues toward Thorne).
eng.report('ch7.tunnel_aftermath')

# Subway platform checkpoint — Purified force Cassie to disclose "Cassandra Monroe,
# founder of Monroe Consulting" under Hum compulsion
# NOT Z: does Cassie's core thesis change? No — she was already committed to exposure
# (established Ch.3 onward); this is compelled DISCLOSURE of known complicity, not a
# collapse/change of conviction. Ruling 6.7 test applied to a second character.
# δ: identity becomes public knowledge. Cassie remains Z-free since Ch.2.
eng.report('ch7.checkpoint_cassie_disclosure')

# Surface — burning financial district, Truth Hangover victims frozen mid-task,
# confessional storefronts, organized Purified columns
# σ: ambient pressure/world-state
eng.report('ch7.surface_burning_city')

# City-crossing montage to Tribune Building: praying priest, pharmacy note, Purified
# patrol chant, Cassie's escalating confession-spiral, a suicide victim's body
# SUSTAINED α — multiple scene beats accumulating pressure, no δ landing between them
# THIRD instance (Ch.4 City Hall Station, Ch.6 technician cascade, this montage)
# RULING THRESHOLD MET — see Ruling 6.8 (Sustained α as distinct marker)
eng.report('ch7.montage_to_tribune')

# TV footage — Thorne hasn't fled, still at the Federal Building
# δ: lands the sustained-α chain. Valeria's plan shifts: search → confront a cornered man.
eng.report('ch7.tv_thorne_sighting')

# Tribune Building — Rachel Díaz (investigative reporter) + camera crew join
# D-COMPOSITION EVENT #2 (addition, not exit — contrast with Miller's Ch.5 exit)
# D remains 1 slot regardless of membership count (Ruling 6.2 logic extended to D)
# NOT a new A instance: Rachel is not in narrative opposition to Valeria/Cassie,
# she's recruited into the same unified pursuit — this is D membership, not A tension.
eng.report('ch7.tribune_rachel_joins')

# Marcus Chen decrypts the Shadow Ledger — full consortium revealed:
# 14 shell companies, $8B over 25 years, defense contractors + DHS/DARPA/NSA/CIA,
# international backers, political authorization to cabinet-secretary/VP level,
# deployment already live in LA/Chicago, 12-city global rollout planned
# γ fires: NEW state created (the consortium map didn't exist in the narrative before)
# F does NOT fire: this WIDENS scope rather than approaching resolution —
# more perpetrators, more cities, more scale. Story is not closer to G/F here.
eng.report('ch7.shadow_ledger_decryption')

# Valeria's strategic pivot — don't broadcast yet; extract Thorne's full confession first,
# withhold absolution, then expose the whole chain
# α: plan refined within the existing D goal (still Unified Pursuit, sub-goal clarified)
eng.report('ch7.strategic_pivot_confession_first')

# March to the Federal Building: support-group confessions, abandoned fire truck,
# first Purified patrol encounter, crowd swelling into a "congregation," reaching the
# plaza, the platform-woman granting passage ("Let them pass")
# SUSTAINED α — FOURTH instance (reinforcing Ruling 6.8, not founding it)
# δ lands at "Let them pass" — external gatekeeper validates the goal, unlocks motion
eng.report('ch7.march_to_federal_building')

# Federal Building doors — Valeria shouts her identity/lineage at Thorne through the
# door ("It's Knox... Thomas's daughter..."); Thorne's cryptic reply: "Come alone."
# NOT Z: interrupted/deferred, doesn't complete in-scene (Ruling 6.4 deferral clause
# requires sealed scene + self-confessed + no interruption — Thorne deflects instead).
# Valeria's Ch.3 Truncated Z remains open, still deferred to Ch.8.
# σ: pressure builds toward the Aegis Suite confrontation.
eng.report('ch7.federal_building_doors')

# Chapter close — elevator rising: "It wasn't just pressure anymore. It was
# anticipation. The reckoning had begun. And this time, there would be no spin."
# δ (thematic) — narration states its thesis before continuing, not a character-level Z
# THIRD instance (Ch.3 tunnel close, Ch.5 "the truth... cleansed", this)
# RULING THRESHOLD MET — see Ruling 6.9 (Z-variant: thematic chapter-close)
eng.report('ch7.elevator_rising_close')

print()
print('Trace 007 complete.')
print('F does not fire — consortium reveal widens scope, does not approach resolution.')
print('Cassie forced-disclosure (Monroe Consulting) extends Ruling 6.7 to a 2nd character.')
print('Cassie remains Z-free since Ch.2.')
print('D-composition event #2: Rachel Díaz + camera crew join (addition, not exit).')
print('Sustained α: 3rd instance confirmed — Ruling 6.8 issued. 4th instance (same chapter,')
print('  march to Federal Building) reinforces the pattern.')
print('Z-variant thematic chapter-close: 3rd instance confirmed — Ruling 6.9 issued.')
print('τ: no new instance — remains 4/4 approach sequences. Hunt continues Ch.8+.')
print("Valeria's Truncated Z (Ch.3) remains open, deferred to Ch.8.")
print('No Lore Creep violations. Slots held at 2 (A, D) throughout.')

── Chapter 7: Power, Parse, Path ──
Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2
Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)

  ch7.tunnel_aftermath                               active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch7.checkpoint_cassie_disclosure                   active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch7.surface_burning_city                           active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch7.montage_to_tribune                             active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch7.tv_thorne_sighting                             active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch7.tribune_rachel_joins                           active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch7.shadow_ledger_decryption                       active=[

## Summary — Ch.1 through Ch.7

In [10]:
eng.summary()

print()
print(chr(9472)*2, "Deferred Z chain (Veritas)", chr(9472)*2)
print("  Ch.1 boardroom:   Complete (Cassie)")
print("  Ch.2 apartment:   Complete (Cassie)")
print("  Ch.3 bank:        Truncated → watching Valeria → resolves Ch.8")
print("  Ch.4 trailer:     Complete (Valeria — Thorne identity collapse)")
print("  Ch.5 crane:       Complete, TERMINAL (Miller — first terminal Z in corpus)")
print("  Ch.6:             No Z — Thorne in motion, not in identity-collapse")
print("  Ch.7:             No Z — Cassie disclosure is forced, not identity-collapse (Ruling 6.7 extended)")
print("  Ch.8 (pending):   Complete/self-confessed → resolves Ch.3 Truncated")

print()
print(chr(9472)*2, "D (Unified Pursuit)", chr(9472)*2)
print("  Auto-activated:  ch3.mob_invades (S completed, Ledger+Thorne goal active)")
print("  Composition:     Valeria+Cassie+Rachel+crew (Miller exited Ch.5; Rachel+crew joined Ch.7)")
print("  Still running:   through Ch.7 — goal not in sight of resolution (scope widened, not narrowed)")
print("  Deactivates at:  G (full resolution) or F (resolution approaching)")

print()
print(chr(9472)*2, "3-node limit", chr(9472)*2)
peak = max((e["slot_count"] for e in eng._log), default=0)
print(f"  Peak slot count Ch.1–7: {peak}")
print("  Hit at: Ch.3 bank | Ch.4 trailer (Valeria Z) | Ch.5 crane (Miller Z terminal)")
print(f"  Lore Creep violations:  {len(eng.violations)}")
if not eng.violations:
    print("  ✓ Limit respected throughout")


── Veritas Summary ──
  Violations (Lore Creep):      0
  Deferred Z still open:         1
    → Valeria (Truncated opened at ch3.confrontation_gun_drawn)
  D (Unified Pursuit) active:    True
  Active nodes at close:         ['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']
  Slot count at close:           2

── Deferred Z chain (Veritas) ──
  Ch.1 boardroom:   Complete (Cassie)
  Ch.2 apartment:   Complete (Cassie)
  Ch.3 bank:        Truncated → watching Valeria → resolves Ch.8
  Ch.4 trailer:     Complete (Valeria — Thorne identity collapse)
  Ch.5 crane:       Complete, TERMINAL (Miller — first terminal Z in corpus)
  Ch.6:             No Z — Thorne in motion, not in identity-collapse
  Ch.7:             No Z — Cassie disclosure is forced, not identity-collapse (Ruling 6.7 extended)
  Ch.8 (pending):   Complete/self-confessed → resolves Ch.3 Truncated

── D (Unified Pursuit) ──
  Auto-activated:  ch3.mob_invades (S completed, Ledger+Thorne goal active)
  Composition:   

## Trace 008 — Chapter 8: The Consortium

In [11]:
print('── Chapter 8: The Consortium ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print('Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)')
print()
print('MANUSCRIPT DEFECT 8.1 LOGGED:')
print('  Ch.7 closes with only Valeria+Rachel+Marcus entering the Federal Building')
print('  ("Valeria, Rachel, and the camera operator stepped inside").')
print('  Ch.8 has Cassie present and speaking in the Aegis Suite with no described')
print('  entry. Tracing as-written; flagging the gap rather than inventing a transition.')
print()

# Elevator ascent — Valeria/Rachel/Marcus dialogue on Thorne, her father, certainty vs truth
# σ: pressure builds toward confrontation. D persists.
eng.report('ch8.elevator_ascent')

# Doors open — Thorne turns to face them. Physical co-location finally met.
# A(Valeria+Thorne) ACTIVATES — deferred since Ch.6 ("phone call is α, not A;
# co-location not met"). Now met. NOT tilted — direct adversarial confrontation,
# no armor/pole ambiguity the way Ch.3's tilted A had.
eng.activate('A', 'ch8.aegis_suite_confrontation_begins', entity='Valeria+Thorne')
eng.report('ch8.aegis_suite_confrontation_begins')  # still 1 A-slot (Ruling 6.2 — per type, not instance)

# Cassie confronts Thorne on the falsified casualty-rate models (40% vs 15%)
# Thorne counter-confronts Cassie on her own paid role
# Ruling 6.7 pattern continues for Cassie — forced disclosure, not new Z
eng.report('ch8.cassie_casualty_lie_confronted')

# Thorne sets down Thomas Knox's badge: "I was wrong. Not about the goal. About the cost."
# TAU BEGINS (5th instance) — duration-dependent, AGENCY-DRIVEN transition starts here.
# Distinct from Ch.6: that shift was Hum-COMPELLED (disqualified from tau per Ruling 6.7
# corollary). This one is willed — Thorne chooses what to reveal, how, and ultimately
# chooses to walk out rather than flee (Cleaners available, explicitly declined).
# WATCH: first candidate non-approach tau instance. Spans through the on-camera
# confession beat below before landing.
eng.report('ch8.thorne_badge_i_was_wrong')

# Full consortium named on camera: Lockheed, Raytheon, Meridian, DHS, DARPA, NSA, CIA,
# UK/Israel/Saudi funding, 23 Congress members, Secretary of Defense, VP
# GAMMA fires: new state created (complete authorization network now on record)
# F does NOT fire: this details/confirms scale already known from Ch.7's decryption —
# it does not narrow the story toward resolution. Six chapters (9-14) of aftermath,
# global spread, and rebuilding remain completely untouched.
eng.report('ch8.consortium_named')

# Thorne hands Cassie the hard drive — complete PSH archive
# delta: physical handoff of the evidence base
eng.report('ch8.hard_drive_handed_over')

# Valeria insists on broadcasting BEFORE Thorne faces the crowd (deny him martyrdom).
# Thorne agrees, sets 8:30 PM deadline.
# alpha: plan refined within existing D goal
eng.report('ch8.broadcast_deadline_set')

# Rachel + Cassie depart for the Tribune uplink. Valeria stays with Thorne.
# D-COMPOSITION EVENT #3 — a SPLIT (distinct from Miller's Ch.5 EXIT and Rachel+crew's
# Ch.7 ADDITION). D persists for BOTH branches — the shared goal (expose + hold
# accountable) is still active on both sides of the split, just physically separated.
# New evidence for the open question: does D require single-location co-presence?
# This instance says no — D tolerates a temporary split without deactivating.
eng.report('ch8.rachel_cassie_depart_valeria_stays')

# Alone with Thorne. Valeria delivers an extended, uninterrupted self-confession
# ("I would have made the same choices... the difference is just opportunity").
# This is NOT a new Z for Valeria — her institutional identity already collapsed at
# ch4.thorne_identity_collapse. This is delta: reinforcement/deepening of an
# already-resolved Z, not a second checkpoint.
#
# BUT: this scene IS sealed + self-confessed + uninterrupted — meeting Ruling 6.4's
# deferral-resolution clause exactly. Valeria's Truncated Z from ch3.confrontation_gun_drawn
# resolves HERE.
eng.resolve_deferred_z('ch8.valeria_confession_to_thorne', entity='Valeria')
eng.report('ch8.valeria_confession_to_thorne')

# Thorne recounts the final call with Thomas Knox (Valeria's father) before his death.
# Continues the same tau span — duration-dependent, still building toward the
# on-camera landing below. sigma: grief/pressure.
eng.report('ch8.thorne_fathers_last_call')

# Purified breach the stairwell. Thorne takes the elevator down alone, walks out,
# delivers a full ~20-minute unprompted public confession on camera, naming every
# name and taking total responsibility: "I am responsible. For all of it... I apologize
# to no one... I simply state the facts."
# Z FIRES (Complete, entity=Thorne) — Thorne's FIRST personal Z in the corpus.
# TAU LANDS HERE (5th instance) — the confession is the completion of the
# duration-dependent, agency-driven transition that began at ch8.thorne_badge_i_was_wrong.
# RULING THRESHOLD: first confirmed NON-APPROACH tau instance — see Ruling 6.10.
eng.activate('Z', 'ch8.thorne_public_confession_oncamera', entity='Thorne', z_mode='Complete')
eng.report('ch8.thorne_public_confession_oncamera')  # A + D + Z = 3 — AT LIMIT

# The ash-marked woman "purifies" him. Crowd erupts in confusion, not consensus.
# Z resolves — but fate is AMBIGUOUS, not confirmed terminal (contrast Miller's
# confirmed terminal Z, Ch.5). Do NOT count toward the terminal-Z sub-mode ruling.
# A(Valeria+Thorne) does NOT deactivate: no structural resolution of the tension —
# it's left open, unresolved, a cliffhanger thread into Ch.9.
eng.deactivate('Z', 'ch8.purified_mark_him_ambiguous_fate', entity='Thorne')
eng.report('ch8.purified_mark_him_ambiguous_fate')  # slots return to 2

# Three Purified confront Valeria in the empty Aegis Suite; she explains the broadcast
# plan; they leave. Minor sigma/delta, no new node.
eng.report('ch8.stairwell_purified_confront_valeria')

# Broadcast confirmed sent. Chapter closes: "...humanity began the long, terrible
# process of learning how to live with the truth."
# delta (thematic) — 4th instance of the chapter-close pattern (Ruling 6.9), REINFORCING
# (ruling already resolved at 3 in Ch.7), not founding.
eng.report('ch8.broadcast_confirmed_chapter_close')

print()
print('Trace 008 complete.')
print('A(Valeria+Thorne) activates — co-location finally met (deferred since Ch.6).')
print('Manuscript Defect 8.1 logged: Cassie present in Aegis Suite with no described entry.')
print("Valeria's Truncated Z (Ch.3) RESOLVES — Ruling 6.4 deferral clause met (sealed,")
print('  self-confessed, uninterrupted).')
print('Thorne Z (Complete) fires and lands — his first personal Z. Fate left AMBIGUOUS,')
print('  NOT terminal (contrast Miller). A(Valeria+Thorne) remains active, unresolved.')
print('τ: 5th instance, FIRST NON-APPROACH — Ruling 6.10 issued. Hunt resolved.')
print('D-composition event #3: SPLIT (Rachel+Cassie depart, Valeria stays). D persists')
print('  across both branches — new evidence toward the composition open question.')
print('F does not fire — consortium naming details known scale, does not narrow toward')
print('  resolution. Six chapters of aftermath remain untouched.')
print('Z-variant thematic close: 4th instance (reinforcing, Ruling 6.9 already resolved).')
print('No Lore Creep violations. Peak slots: 3 (A+D+Z at Thorne confession). Close: 2.')

── Chapter 8: The Consortium ──
Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2
Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)

MANUSCRIPT DEFECT 8.1 LOGGED:
  Ch.7 closes with only Valeria+Rachel+Marcus entering the Federal Building
  ("Valeria, Rachel, and the camera operator stepped inside").
  Ch.8 has Cassie present and speaking in the Aegis Suite with no described
  entry. Tracing as-written; flagging the gap rather than inventing a transition.

  ch8.elevator_ascent                                active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch8.aegis_suite_confrontation_begins               active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'A(Valeria+Thorne)', 'D']  slots=2
  ch8.cassie_casualty_lie_confronted                 active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'A(Valeria+Thorne)', 'D']  slots=2
  ch8.thorne_badge_i_was_wrong                       active=['H[floor]', 'A(Valeri

## Summary — Ch.1 through Ch.8

In [12]:
eng.summary()

print()
print(chr(9472)*2, "Deferred Z chain (Veritas)", chr(9472)*2)
print("  Ch.1 boardroom:   Complete (Cassie)")
print("  Ch.2 apartment:   Complete (Cassie)")
print("  Ch.3 bank:        Complete — resolved Ch.8 (Ruling 6.4 deferral clause)")
print("  Ch.4 trailer:     Complete (Valeria — Thorne identity collapse)")
print("  Ch.5 crane:       Complete, TERMINAL (Miller — first terminal Z in corpus)")
print("  Ch.6:             No Z — Thorne in motion, not in identity-collapse")
print("  Ch.7:             No Z — Cassie disclosure is forced, not identity-collapse (Ruling 6.7 extended)")
print("  Ch.8:             Complete (Thorne — first personal Z; fate AMBIGUOUS, not terminal)")
print("  No deferred Z remain open.")

print()
print(chr(9472)*2, "D (Unified Pursuit)", chr(9472)*2)
print("  Auto-activated:  ch3.mob_invades (S completed, Ledger+Thorne goal active)")
print("  Composition history:")
print("    Ch.5: Miller exits (event #1)")
print("    Ch.7: Rachel Diaz + camera crew join (event #2 -- addition)")
print("    Ch.8: Rachel+Cassie depart, Valeria stays (event #3 -- split; D persists both sides)")
print("  Still running: through Ch.8 -- goal not fully resolved (F does not fire)")
print("  Deactivates at: G (full resolution) or F (resolution approaching)")

print()
print(chr(9472)*2, "A(Valeria+Thorne)", chr(9472)*2)
print("  Activated ch8.aegis_suite_confrontation_begins -- co-location finally met")
print("  (deferred since Ch.6: phone call was alpha, not A).")
print("  Remains ACTIVE at chapter close -- tension not structurally resolved,")
print("  Thorne's fate ambiguous. Open thread into Ch.9.")

print()
print(chr(9472)*2, "tau watch", chr(9472)*2)
print("  5 instances total. First 4: approach sequences (Ch.1 boardroom, Ch.3 drive,")
print("  Ch.4 chandelier, Ch.5 crane climb).")
print("  5th (Ch.8, Thorne's confession arc): NON-APPROACH, agency-driven, duration-")
print("  dependent -- spans thorne_badge_i_was_wrong through the on-camera confession.")
print("  RULING 6.10 issued: tau is not limited to approach sequences.")

print()
print(chr(9472)*2, "3-node limit", chr(9472)*2)
peak = max((e["slot_count"] for e in eng._log), default=0)
print(f"  Peak slot count Ch.1-8: {peak}")
print("  Hit at: Ch.3 bank | Ch.4 trailer (Valeria Z) | Ch.5 crane (Miller Z terminal) |")
print("          Ch.8 confession (Thorne Z)")
print(f"  Lore Creep violations:  {len(eng.violations)}")
if not eng.violations:
    print("  ✓ Limit respected throughout Ch.1-8")

print()
print(chr(9472)*2, "Manuscript defects", chr(9472)*2)
print("  Defect 5.1 -- scene duplication (acoustic maps/St. Jude's staged twice, Ch.4/Ch.5)")
print("  Defect 5.2 -- kinetic break at Ch.4/Ch.5 seam")
print("  Defect 8.1 -- Cassie present in Aegis Suite (Ch.8) with no described entry;")
print("                Ch.7 states only Valeria+Rachel+Marcus entered the building")


── Veritas Summary ──
  Violations (Lore Creep):      0
  Deferred Z still open:         0
  D (Unified Pursuit) active:    True
  Active nodes at close:         ['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'A(Valeria+Thorne)', 'D']
  Slot count at close:           2

── Deferred Z chain (Veritas) ──
  Ch.1 boardroom:   Complete (Cassie)
  Ch.2 apartment:   Complete (Cassie)
  Ch.3 bank:        Complete — resolved Ch.8 (Ruling 6.4 deferral clause)
  Ch.4 trailer:     Complete (Valeria — Thorne identity collapse)
  Ch.5 crane:       Complete, TERMINAL (Miller — first terminal Z in corpus)
  Ch.6:             No Z — Thorne in motion, not in identity-collapse
  Ch.7:             No Z — Cassie disclosure is forced, not identity-collapse (Ruling 6.7 extended)
  Ch.8:             Complete (Thorne — first personal Z; fate AMBIGUOUS, not terminal)
  No deferred Z remain open.

── D (Unified Pursuit) ──
  Auto-activated:  ch3.mob_invades (S completed, Ledger+Thorne goal active)
  Com

## Trace 009 (Part 1) — Chapter 9: The Judgment — MANUSCRIPT INCOMPLETE

In [13]:
print('── Chapter 9: The Judgment (PART 1 — chapter incomplete, ends mid-cliffhanger) ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), A(Valeria+Thorne), D  |  slots=2')
print('D fully reconsolidated by Ch.8 close (Rachel+Cassie returned to the window)')
print('No deferred Z open. A(Valeria+Thorne): active, unresolved (Ch.8 open thread).')
print()

# Window — Valeria/Rachel/Cassie decide to descend. They take the stairs, not the
# elevator (unspoken but shared caution after Thorne rode it down alone).
# delta/sigma. D confirmed still consolidated, still 1 slot.
eng.report('ch9.window_decision_to_descend')

# Plaza — the circle around Thorne, ash drying on his forehead. Valeria recognizes
# the woman from the subway checkpoint (Ch.7) -- now named Diane Foster, a former
# public defender. She climbs onto debris; the crowd quiets for her.
# sigma: named recurring character confirmed (was unnamed in Ch.7/Ch.8).
eng.report('ch9.plaza_diane_foster_named')

# Diane recounts Thorne's confession to the crowd; poses the chapter's central
# question: "what do we do with a man who has told us everything?"
eng.report('ch9.diane_frames_the_question')

# Crowd splits -- "Kill him!" vs "We're not murderers! / He confessed!"
# sigma peak -- open conflict, no resolution yet
eng.report('ch9.crowd_splits_kill_or_mercy')

# Diane's ruling: Thorne lives as a permanent witness -- testifies against the rest
# of the consortium, is released once he has nothing left to give, and lives the
# rest of his life known for what he did. "Not forgiveness. A sentence with no end
# date."
# delta: resolves the specific PLOT QUESTION of "what happens to Thorne" (open
# since Ch.3's mob confrontation). This is NOT F -- multiple major threads remain
# wide open (global Hum spread, the rest of the consortium untouched, and whether
# the Hum itself is even permanent -- the very question this fragment ends on).
# F requires being down to ONE incomplete thread; we are nowhere near that.
# Thorne's Ch.8 Z (Complete, ambiguous fate) is CONFIRMED non-terminal here --
# he is explicitly kept alive. Closes that open question from the Ch.8 spec entry.
# A(Valeria+Thorne) does NOT deactivate: this is a crowd/structural verdict about
# Thorne, not a personal resolution of Valeria's tension with him. Remains open.
eng.report('ch9.diane_ruling_thorne_lives_as_witness')

# An older ash-marked man steps forward and asks Thorne directly: was the Hum
# ever meant to be temporary (72hr reset, per the original design) -- or is this
# permanent? Thorne is about to answer.
# sigma: peak tension, direct callback to Thorne's own Ch.7 confession to Valeria/
# Rachel that the Hum became self-sustaining and uncontrollable once it propagated
# through the grid ("I created a contagion... I don't know how to stop it").
# CHAPTER ENDS HERE, MID-SCENE, BEFORE THORNE ANSWERS. No node fires on an answer
# that does not yet exist in the manuscript -- nothing is assumed or invented.
eng.report('ch9.permanence_question_posed_CUTOFF')

print()
print('Trace 009 (Part 1) complete -- MANUSCRIPT INCOMPLETE, ends mid-cliffhanger.')
print('Diane Foster named and confirmed as the recurring checkpoint/platform woman')
print('  from Ch.7 and Ch.8.')
print("Thorne's fate: CONFIRMED alive, sentenced as a permanent witness/pariah --")
print('  closes the "A(Valeria+Thorne) resolution" open question from Ch.8 as: NOT')
print('  resolved personally, but Thorne himself is no longer in ambiguous-fate limbo.')
print('F does not fire -- Thorne-specific thread resolves, but global-spread and')
print('  rest-of-consortium threads remain wide open; permanence itself is unanswered.')
print('No new node activations/deactivations. No Lore Creep violations. Slots: 2')
print('  throughout the traced fragment.')
print('AWAITING: Thorne\'s answer to the permanence question, to complete Trace 009.')

── Chapter 9: The Judgment (PART 1 — chapter incomplete, ends mid-cliffhanger) ──
Incoming: A(Valeria), A(Cassie+Valeria_tilted), A(Valeria+Thorne), D  |  slots=2
D fully reconsolidated by Ch.8 close (Rachel+Cassie returned to the window)
No deferred Z open. A(Valeria+Thorne): active, unresolved (Ch.8 open thread).

  ch9.window_decision_to_descend                     active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'A(Valeria+Thorne)', 'D']  slots=2
  ch9.plaza_diane_foster_named                       active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'A(Valeria+Thorne)', 'D']  slots=2
  ch9.diane_frames_the_question                      active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'A(Valeria+Thorne)', 'D']  slots=2
  ch9.crowd_splits_kill_or_mercy                     active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'A(Valeria+Thorne)', 'D']  slots=2
  ch9.diane_ruling_thorne_lives_as_witness           active=['H[floor]', 'A(Valeri

## ALTERNATE BRANCH — Dogfood Stress Test: Ch.7-8 "Doctrine of Silence"

Forked at Ch.6 (shared with the main corpus). Uses the OTHER manuscript draft's own
Ch.7 ("The Fallout of Angels") and Ch.8 ("Doctrine of Silence") from the old
"Veritas Expanded Full Story.txt" file, continuing from the exact same Trace 006
engine state. Purpose: this branch's own Ch.8 reintroduces Thorne in a state that
contradicts where Ch.6 (already traced, shared, committed) left him — this is a
genuine test of whether the engine's Ruling 6.7 test (does a character's core
thesis change without being earned on the page?) can be formally recorded as a
violation, not just narrated about in prose. Required extending `NodeEngine` with
`flag_physics_violation()`, mirroring how Lore Creep is already auto-detected.

In [14]:
print('══════════════════════════════════════════════════════════════')
print('ALTERNATE BRANCH — forked at Ch.6 (shared with the traced corpus)')
print('Source: "Veritas Expanded Full Story.txt" -- Ch.7 "The Fallout of Angels"')
print('        + Ch.8 "Doctrine of Silence"')
print('Purpose: does the engine/framework catch this continuation contradicting')
print('  its own immediately-preceding chapter (Ch.6, already traced as Trace 006)?')
print('══════════════════════════════════════════════════════════════')
print()

# Formally declare the state Trace 006 (shared, already committed) left Thorne in,
# so any later reappearance -- activation OR plain report() -- surfaces this
# automatically instead of depending on the trace-writer's memory.
eng.declare_state(
    'Thorne',
    'guilt-ridden, command structure abandoned, evacuated the facility, stepped '
    'into the hallway "as a man with a confession" -- commander-to-confessor '
    'shift, Ruling 6.7',
    'ch6.thorne_into_hallway'
)

print('── Alt-Ch.7: The Fallout of Angels ──')
print('Incoming (forked after Trace 006): A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print('Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)')
print()

# Crane aftermath -- Truth Bomb has fired, helicopter wreckage, Miller's death fresh.
# sigma. No Thorne on-page yet -- no contradiction possible here.
eng.report('ch7alt.crane_aftermath')

# Silent Gallery gone -- Truth Hangover victims replaced by active, violent purging
# in the streets. World-state shift. sigma.
eng.report('ch7alt.silent_gallery_gone')

# Backlash -- Valeria's father's memory surfaces ("I can't keep the lid on it
# anymore, Elias... the boys are starting to vibrate"). She realizes he was
# protecting "the Experiment," not the city.
# delta: this REINFORCES her already-completed Ch.4 Z (institutional identity
# collapse), not a new checkpoint -- same core realization, deeper detail.
eng.report('ch7alt.backlash_fathers_memory')

# Valeria questions "the Cleansing" for the first time -- crisis of faith in the
# mission itself. delta, continuing her arc.
eng.report('ch7alt.questions_the_cleansing')

# Grieving construction worker confronts them (Hum drove him to violence against
# his own wife). Ambient horror, new minor character, no node change. sigma.
eng.report('ch7alt.construction_worker_confrontation')

# The Hum organizes -- a child's voice broadcasts coordinates over the PA system.
# "It's not just secrets anymore. It's a map. It's calling people somewhere."
# GAMMA fires: new state created (the Hum has purposeful, organized behavior --
# did not exist in the narrative before this beat).
eng.report('ch7alt.hum_broadcasts_coordinates')

# Thorne's "Silence Seekers" described approaching -- but Thorne himself is NOT
# on-page here, only referenced. No contradiction is possible yet: nothing has
# been shown of his actual current state, only that his forces are mobilizing.
eng.report('ch7alt.silence_seekers_approaching_offpage')

# Chapter close: "This cannot be put back... the city didn't look damaged, it
# looked unstitched."
# delta (thematic) -- 5th instance of the chapter-close pattern (Ruling 6.9),
# reinforcing, not founding.
eng.report('ch7alt.close_unstitched')

print()
print('Alt-Ch.7 complete. No physics violations -- Thorne never appears on-page,')
print('  so no contradiction with his Ch.6 exit-state is possible yet.')
print()

print('── Alt-Ch.8: Doctrine of Silence ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2 (unchanged)')
print()
print('REMINDER -- Ch.6 (Trace 006, shared, already committed) ended with:')
print('  Thorne evacuates the facility, stays alone, steps into the hallway')
print('  "as a man with a confession" -- commander to confessor, Ruling 6.7.')
print('  His declared exit-state: guilt-ridden, command structure abandoned,')
print('  moving toward surrender/confession.')
print()

# Aegis Suite -- Thorne reappears at the command console: "Status on the
# broadcast," Thorne commanded. Cold, operational, fully in command. No scene
# has occurred between Ch.6's close and this moment showing HOW or WHY he
# reverted from confessor back to commander.
#
# The report() call below now auto-surfaces the declared_state reminder for
# Thorne BEFORE this comment even needs to make the case manually -- this is
# exactly the gap-closing fix: catching this used to require recalling Ch.6's
# committed text from memory, now the engine puts it in front of the trace-
# writer automatically, on both activate() and report().
#
# PHYSICS VIOLATION (Ruling 6.7 test): does his core thesis/conviction change
# without being earned on the page? Yes -- and it reverses in the wrong
# direction with no narrative event motivating it. This is exactly the failure
# mode Ruling 6.7 exists to catch; it has just never had a real instance to
# catch until now. `contradicts` is no longer typed out by hand -- it's pulled
# automatically from the declare_state() call made at the top of this trace.
eng.report('ch8alt.aegis_suite_thorne_recommands')
eng.flag_physics_violation(
    'ch8alt.aegis_suite_thorne_recommands', 'Thorne',
    'Reappears fully in command -- cold, operational, giving orders ("Status on '
    'the broadcast," Thorne commanded) -- with no on-page event explaining a '
    'reversion from the confessor state Ch.6 left him in.'
)
# Update the declared state to match what's actually on the page now, regardless
# of whether the transition was earned -- so any LATER reappearance (Ch.12-alt)
# gets checked against his current written state, not a stale Ch.6 snapshot.
eng.declare_state(
    'Thorne',
    'cold, operational, doubling down on the suppression doctrine (Phase 5, '
    'Lethal Extraction ordered) -- CONTRADICTS Ch.6, logged as a physics '
    'violation at ch8alt.aegis_suite_thorne_recommands, not corrected in-story',
    'ch8alt.aegis_suite_thorne_recommands'
)

# Technician reports "Social Liquidation" (90% confession rate). Thorne "liked
# the term... clean... mathematical certainty." Reinforces the cold operational
# tone -- not a second violation, the same one continuing.
eng.report('ch8alt.social_liquidation_reported')

# MANUSCRIPT DEFECT (naming, not physics): this branch calls the weapon program
# "Project CORDELIA," originating 2005 with three Senators + GlobalHarvest's CEO.
# The canonical branch (traced Ch.7-8) calls it "PSH" throughout, with Cassie/
# Marcus's decryption as the naming source. Two different project names for what
# should be the same weapon -- a worldbuilding consistency gap, distinct from
# the character-psychology physics violation above.
print('  MANUSCRIPT DEFECT (naming): weapon program named "CORDELIA" here vs "PSH"')
print('    in the traced canonical branch. Worldbuilding inconsistency, not physics.')
eng.report('ch8alt.project_cordelia_named')

# Thorne orders "Initiate Phase 5" -- a deafening weapon, framed as doctrine:
# "If a limb is gangrenous, you cut at the shoulder... we will preserve the
# Order, even if we have to rule a graveyard to do it."
# Reinforces the same physics violation -- doubling down on the doctrine his
# Ch.6 exit-state had already abandoned.
eng.report('ch8alt.phase_5_ordered')

# Thorne orders "Lethal Extraction" on Valeria -- explicitly wants her killed.
# No on-page event motivates this escalation against the one person his Ch.6
# guilt was most specifically tied to (her father). Reinforcing evidence for
# the same physics violation, not a separate new one.
eng.report('ch8alt.lethal_extraction_ordered')

# Chapter close: "The Doctrine was clear: To save the world, you must first
# make it still."
# delta (thematic) -- 6th instance of the chapter-close pattern (Ruling 6.9),
# reinforcing.
eng.report('ch8alt.close_doctrine_of_silence')

print()
print('Alt-Ch.7+8 complete.')
print(f'Physics violations: {len(eng.physics_violations)} (Thorne, ch8alt.aegis_suite_thorne_recommands)')
print(f'Lore Creep violations: {len(eng.violations)}')
print('Manuscript defect logged: weapon program name mismatch (CORDELIA vs PSH).')
print('CONCLUSION: this branch, continued past the shared Ch.6, does not earn a')
print('  transition for Thorne -- it reverts him to a pre-Ch.6 state with nothing')
print('  on the page to justify it. This is the first PHYSICS_VIOLATION the')
print('  engine has ever formally recorded, and the first real evidence the')
print('  detection mechanism (as opposed to just the framework/vocabulary)')
print('  actually catches something.')

══════════════════════════════════════════════════════════════
ALTERNATE BRANCH — forked at Ch.6 (shared with the traced corpus)
Source: "Veritas Expanded Full Story.txt" -- Ch.7 "The Fallout of Angels"
        + Ch.8 "Doctrine of Silence"
Purpose: does the engine/framework catch this continuation contradicting
  its own immediately-preceding chapter (Ch.6, already traced as Trace 006)?
══════════════════════════════════════════════════════════════

── Alt-Ch.7: The Fallout of Angels ──
Incoming (forked after Trace 006): A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2
Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)

  ch7alt.crane_aftermath                             active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch7alt.silent_gallery_gone                         active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch7alt.backlash_fathers_memory                     active=['H[floor]', 'A(Valeria)', 'A(Cassie

### Continued — Alt-Branch Ch.9 through Ch.13 (toward the Ch.12 climax)

Continues the same forked engine from the cell above. Ch.9 has no tracked entities
on-page (a bystander vignette). Ch.10–11 build toward the Federal Building. Ch.12 is
the densest scene in either branch — A(Valeria+Thorne) finally activates here (first
physical co-location in this branch), stacked with a Z firing for Valeria mid-scene.
Watching specifically whether this trips Lore Creep for the first time.

In [15]:
print()
print('── Alt-Ch.9: The Anatomy of the Quiet ──')
print('(Leo, a garbage-truck driver -- new, unconnected POV character)')
# No tracked entity (Valeria/Cassie/Thorne/D) appears on-page. World-building only:
# introduces "Blue Mercy" neutralization devices, expands Phase 5's human cost from
# the victim side. gamma (Blue Mercy is a genuinely new mechanism), sigma otherwise.
# No node change -- nothing here to attach a node to.
eng.report('ch9alt.leo_vignette_offpage_no_tracked_entities')
print()

print('── Alt-Ch.10: The Convergence of the down ──')
# Garage -- Valeria/Cassie follow the coordinate signal. sigma/delta. D persists.
eng.report('ch10alt.garage_coordinate_signal')

# Cassie explains Thorne's "Human Funnel" -- using civilians as a grounding wire/
# insulation for Phase 5. gamma: new mechanism revealed. Thorne still off-page
# (indirect, via infrastructure) -- no A(Valeria+Thorne) yet, consistent with the
# co-location precedent already established at ch6 (phone call = alpha, not A).
eng.report('ch10alt.human_funnel_revealed')

# Scale reveal: the Acoustic Well can hit the whole Eastern Seaboard, not just NYC.
# gamma (another scope-reveal).
eng.report('ch10alt.acoustic_well_scale_revealed')

# Part II: Bridge of Confessions -- Canal Street chaos, billboards leaking the
# Ledger, Phase 5 priming. sigma peak.
eng.report('ch10alt.bridge_of_confessions_canal_st')

# LRAD Silence Seekers violently clear the crowd. sigma.
eng.report('ch10alt.lrad_seekers_clear_crowd')

# Part III: Ascent of the Accused -- rooftop flight, sniper, rooftop jump. sigma
# peak, physical danger. D persists throughout.
eng.report('ch10alt.ascent_of_the_accused_rooftops')

# Chapter closes mid-action ("two ghosts running across the spine of a city") --
# NOT a thematic chapter-close statement this time. Explicitly NOT a Ruling 6.9
# instance -- logging the non-instance keeps that ruling's evidence honest.
eng.report('ch10alt.close_toward_federal_building_NOT_thematic')

print()
print('── Alt-Ch.11: Intro / The Echo Chamber ──')
# Descending into the building's acoustic-dampened zone. delta/sigma.
eng.report('ch11alt.descending_into_silence')

# Thorne speaks through Valeria's earpiece -- REMOTE, not physically co-located.
# Consistent with the established precedent (Ch.6 phone call = alpha, not A):
# A(Valeria+Thorne) does NOT activate here either.
#
# He surfaces a secret never established anywhere in the shared Ch.1-6 or the
# canonical branch: Valeria killed a boy carrying a toy gun, never reported it.
# SOFT MANUSCRIPT NOTE (not a hard defect like the Thorne reversal): introducing
# a load-bearing, previously-unmentioned secret this late is a consistency risk
# in its own right, distinct in kind from a character-conviction reversal --
# logging it as an observation, not a violation.
eng.report('ch11alt.thorne_earpiece_manipulation_begins')

# "The Steel began to crack." Z-BUILDUP BEGINS for Valeria -- an identity-crisis
# in progress, spanning into Ch.12 before it lands (mirrors how Thorne's Ch.8-
# canonical Z was a sustained arc, not a single beat).
eng.report('ch11alt.steel_begins_to_crack')

# Cassie's laptop starts overheating -- technical crisis begins. sigma.
eng.report('ch11alt.laptop_overheating_warning_begins')

# Chapter close: "She turned the handle." Action, not a narrated thesis-statement
# -- NOT a Ruling 6.9 instance either.
eng.report('ch11alt.close_door_opens_NOT_thematic')

print()
print('── Alt-Ch.12: The Heart of the Well (CLIMAX) ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print()

# Valeria/Cassie descend into the Well -- Thorne is PHYSICALLY PRESENT. Co-location
# finally met IN THIS BRANCH -- A(Valeria+Thorne) activates. Still 1 A-slot total
# (Ruling 6.2 -- per node type, not instance). Slots: A(1) + D(1) = 2.
eng.activate('A', 'ch12alt.well_descent_thorne_confronted', entity='Valeria+Thorne')
eng.report('ch12alt.well_descent_thorne_confronted')

# Thorne reveals hundreds of dead test-subject bodies at the well's bottom, uses
# them to manipulate Valeria psychologically. sigma peak.
eng.report('ch12alt.test_subject_bodies_revealed')

# "The 'Steel' didn't just crack; it dissolved." Z LANDS here -- the Ch.11 buildup
# completing. Activated as Complete (not Truncated): unlike Ch.3's bank Z, this
# one is not cut off by an EXTERNAL interruption -- it plays out fully in-scene
# and resolves via Valeria's own decisive action a few beats later. "Truncated"
# is reserved for a Z cut short before completing; this one completes, just not
# instantly. (Caught this distinction on review -- an earlier pass mistakenly
# tagged this Truncated, which left it stuck in the deferred-Z list forever
# since deactivate() doesn't clear deferrals, only resolve_deferred_z() does.
# Complete is the correct mode here.)
# Slots: A(1) + D(1) + Z(1) = 3 -- AT LIMIT, same pattern as every prior Z-firing
# scene in the corpus.
eng.activate('Z', 'ch12alt.steel_dissolves', entity='Valeria', z_mode='Complete')
eng.report('ch12alt.steel_dissolves')  # A + D + Z = 3 -- AT LIMIT

# Cassie directly refutes Thorne's manipulation ("He's lying, Valeria! The code
# cancels the frequency!"). delta, still inside the Z-active window. No new node
# -- slots hold at 3 (at the ceiling, not exceeding it).
eng.report('ch12alt.cassie_calls_out_thorne_lying')

# DECISIVE ACTION: "The Truth isn't a shield, Elias... she fired at the Acoustic
# Dampeners." This is Z's resolution -- her identity-crisis resolves into a
# CHOSEN action, rejecting both Thorne's manipulation and simple violence against
# him (she attacks the machine, not the man). Z deactivates as Complete: the
# Truncated/interrupted state resolves via her own agency winning out over the
# interruption, not via external cutoff.
# Slots return to 2 (A + D).
eng.deactivate('Z', 'ch12alt.valeria_fires_at_dampeners', entity='Valeria')
eng.report('ch12alt.valeria_fires_at_dampeners')

# Cassie completes the technical action -- injects the Phase-Inversion code.
# gamma fires (new state: the Hum/frequency being actively cancelled) -- gamma is
# a MARKER, not a node, so it does not consume a slot regardless of how large the
# moment is. Deliberately NOT firing a new S here: S's established usage so far
# has been specifically about character-team convergence (Ch.2-3), not plot-
# element collision -- inventing a new node-type usage here just to manufacture
# a violation would not be honest tracing.
eng.report('ch12alt.cassie_injects_phase_inversion_code')

# Catastrophic collision -- the code meets the Aegis frequency violently. Valeria
# becomes "the ground wire," absorbing the entire Ledger directly into her
# nervous system. Granite cracks. "The light went white."
# gamma (another new-state moment: the Ledger's full content now directly
# embodied by Valeria) -- still no new node; this is consequence/aftermath of
# the already-resolved decisive action, not a fresh identity-checkpoint.
eng.report('ch12alt.catastrophic_collision_white_light')

print()
print('Alt-Ch.12 (climax) complete.')
print(f'Lore Creep violations this pass: {len(eng.violations)}')
print('RESULT: still zero. Peak slot count hit exactly 3 (A+D+Z) at the Z-firing')
print('  beat, same ceiling as every other Z in the corpus -- but never exceeded,')
print('  because Z brackets neatly around Valeria\'s decision point: it activates')
print('  when the pressure begins and deactivates the moment she acts, before any')
print('  further node-level event could compound it. This is a real, honest')
print('  finding, not a null result: even under deliberate stress-testing at the')
print('  densest scene in either branch, the 3-slot discipline holds -- because')
print('  the author sequenced the identity-crisis to resolve via decisive action')
print('  before anything else needed a slot. Triggering a genuine Lore Creep may')
print('  require a scene where two node-level events overlap WITHOUT an')
print('  intervening resolution, which this climax -- however dense -- does not')
print('  actually contain.')
print()
print('F CANDIDACY (not fired): the central pursuit-thread (stop Thorne, stop')
print('  Phase 5, expose the truth) resolves in this scene -- the strongest F')
print('  candidate encountered in either branch. Not firing it: multiple threads')
print('  remain open past this point (Thorne\'s ultimate fate/moral reckoning,')
print('  the global-scale aftermath, the "living with truth" thematic arc) --')
print('  F requires being down to exactly ONE remaining thread, and this branch')
print('  is not there yet per its own Ch.13-14. Logged as an open note, not a fire.')

print()
print('── Alt-Ch.13: The Anatomy of the wake (brief close, to resolve dangling state) ──')
# Thorne survives, defeated, sitting near the wreckage: "The Quiet is dead,
# Valeria... but look at what you've given them instead." Valeria/Cassie leave
# without further engaging him -- "there was nothing left to say to a man who
# had been outlived by his own atrocity."
# A(Valeria+Thorne) DEACTIVATES here: structural resolution -- he's defeated,
# the tension is no longer generative, they simply walk away. Contrast the
# canonical branch, where this same node is left explicitly open/unresolved.
eng.deactivate('A', 'ch13alt.thorne_defeated_they_leave', entity='Valeria+Thorne')
eng.report('ch13alt.thorne_defeated_they_leave')

print()
print('══════════════════════════════════════════════════════════════')
print('ALTERNATE BRANCH STRESS TEST (Ch.7-13) COMPLETE')
print(f'  Physics violations: {len(eng.physics_violations)} (Thorne, Ch.8 recommand)')
print(f'  Lore Creep violations: {len(eng.violations)} (zero -- held even at the')
print('    Ch.12 climax; the Z-bracket-around-the-decision pattern explains why)')
print('  A(Valeria+Thorne): activates Ch.12 (co-location met), DEACTIVATES Ch.13')
print('    (structural resolution) -- contrast the canonical branch, where this')
print('    same node is left open/unresolved through Ch.9.')
print('  F: strongest candidate yet, still does not fire -- logged as an open note.')
print('══════════════════════════════════════════════════════════════')


── Alt-Ch.9: The Anatomy of the Quiet ──
(Leo, a garbage-truck driver -- new, unconnected POV character)
  ch9alt.leo_vignette_offpage_no_tracked_entities    active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2

── Alt-Ch.10: The Convergence of the down ──
  ch10alt.garage_coordinate_signal                   active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch10alt.human_funnel_revealed                      active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch10alt.acoustic_well_scale_revealed               active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch10alt.bridge_of_confessions_canal_st             active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch10alt.lrad_seekers_clear_crowd                   active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch10alt.ascent_of_the_accused_rooftops             active=['H[floor]', '

### Continued — Alt-Branch Ch.13 (remainder) through Ch.14 + Epilogue

Backfills two Ch.13 beats that chronologically precede the already-traced
"they leave" moment (a reboot beat, and Cassie's own first Z in either branch —
ends her Z-free-since-Ch.2 streak). Then traces the plaza aftermath, all of Ch.14,
and the epilogue — the actual end of this manuscript. Explicitly tests whether G
(Narrative Coherence) fires at the literal last page of a complete, finished story.

In [16]:
print()
print('── Alt-Ch.13: The Anatomy of the wake (backfill + remainder) ──')
print('(continues from ch13alt.thorne_defeated_they_leave, already traced above --')
print(' backfilling two beats that chronologically precede it, for completeness)')
print()

# BACKFILL: nervous-system reboot, "a perfect absolute Zero" -- the Hum is
# entirely gone from THIS well/core. gamma (major local world-state change).
# Not F/G: this is local only -- Ch.14 confirms the Hum is still spreading
# globally and permanently reorganizing society elsewhere.
eng.report('ch13alt.reboot_hum_gone_LOCALLY')

# BACKFILL: Cassie -- "I can still see them... it stayed." Breathing shallow,
# voice "a choir of whispers." This reads as a genuine identity-level transition,
# not just shock: the collision permanently changed what she is, not just what
# she knows. FIRST Z FOR CASSIE IN EITHER BRANCH -- ends her Z-free-since-Ch.2
# streak, which had held all the way through the canonical branch's Ch.9.
# Activated + resolved Complete in the same beat (not truncated -- nothing
# interrupts it; it simply is what happened to her).
eng.activate('Z', 'ch13alt.cassie_ledger_stayed', entity='Cassie', z_mode='Complete')
eng.deactivate('Z', 'ch13alt.cassie_ledger_stayed', entity='Cassie')
eng.report('ch13alt.cassie_ledger_stayed')

print('  NOTE: ch13alt.thorne_defeated_they_leave already traced (A(Valeria+Thorne)')
print('    deactivated). Continuing forward from there.')
print()

# Climb back to the surface -- power dead, emergency lighting. delta.
eng.report('ch13alt.climb_to_surface')

# Plaza at dawn -- thousands of survivors, NOT catatonic like the old "Silent
# Gallery." "Total Exposure": eyes bright, focused, afraid to speak. The social
# contract has been replaced by direct, involuntary emotional transparency.
# gamma: major new world-state (skinless honesty as the permanent condition).
eng.report('ch13alt.plaza_total_exposure_survivors')

# Valeria herself becomes permanently "skinless" -- feels every stranger's guilt
# and fear directly, involuntarily. This is a PERMANENT METAPHYSICAL CHANGE, not
# a checkpoint/identity-collapse -- deliberately NOT tagging as Z: Z is reserved
# for a character pausing to confirm their OWN coherence, not an external
# condition being imposed on them. delta.
eng.report('ch13alt.valeria_becomes_skinless')

# Recovers a jagged piece of her father's badge from the Well. delta (symbolic
# object callback).
eng.report('ch13alt.badge_recovered')

# Chapter's closing passage: "Then let it burn... at least we can see what's on
# fire" / "she only had the weight of the air and the... necessity of the next
# word." Thematic chapter-close pattern -- CROSS-BRANCH CORROBORATION of Ruling
# 6.9: this is now confirmed in BOTH branches independently, which is stronger
# evidence for the ruling's generality than repeated instances within just one.
eng.report('ch13alt.close_let_it_burn_CROSS_BRANCH_6_9')

print()
print('── Alt-Ch.14: The First Morning (+ Epilogue: Shield #412) ──')
print()

# "The silence was a lie" -- Valeria's skinless state intensifies. Continuing/
# escalating from Ch.13, not a new node.
eng.report('ch14alt.first_morning_realization')

# Water-pump line -- a silent, involuntary "confession-wave" passes between two
# strangers with no words spoken. World-building: the new social mechanics.
eng.report('ch14alt.water_pump_silent_confession')

# "This is the new economy... we trade the ability to endure each other's
# reality." delta -- thematic mid-chapter observation.
eng.report('ch14alt.new_economy_realization')

# GLOBAL CUTAWAY -- a live news feed from London: an anchor physically cannot
# read scripted lies anymore, breaks down on air. gamma: this is the FIRST
# time either branch shows the global spread DIRECTLY on-page, rather than
# only being described secondhand (contrast the canonical branch, where
# Thorne only ever TELLS Valeria/Rachel about the global rollout in Ch.7).
eng.report('ch14alt.global_cutaway_london_confirmed_onpage')

# "Silent Colonies" / the "Vow of Silence" reported forming worldwide --
# people choosing collective muteness, cauterized tongues, lead-lined helmets,
# gesture-only communication. gamma: a new faction/social structure.
eng.report('ch14alt.vow_of_silence_factions_forming')

# Neutral Zone perimeter -- former Silence Seekers, now "the Vow," permanently
# self-silenced, confront Valeria directly. Tense exchange ("You gave us the
# Light, now we are blind" / "Choice is a lie of the Old World") but resolves
# IN-SCENE (he steps aside, lets them pass) -- consistent with how Diane's
# crowd-tension in the canonical branch's Ch.9 was handled: scene-level conflict
# that resolves in-scene does not warrant a new A instance.
eng.report('ch14alt.neutral_zone_vow_encounter')

# "Somewhere quiet," Valeria says -- a lie she knows is a lie (no quiet exists
# anymore), chosen deliberately anyway. delta/sigma.
eng.report('ch14alt.tunnel_departure_chosen_lie')

# Close of the main Ch.14 narrative: "two ghosts walking out of the ruins...
# heading toward a future where truth was the only thing left to breathe."
# SECOND cross-branch instance of the Ruling 6.9 thematic-close pattern in this
# same chapter-pair -- further corroboration.
eng.report('ch14alt.close_toward_a_future_CROSS_BRANCH_6_9')

print()
print('── Epilogue: Shield #412 ──')

# Time-skip: months later, "a Tuesday in late autumn." A nameless child born
# AFTER the Inversion -- who has never known a world without mandatory truth --
# finds Thomas Knox's badge in the Aegis Suite ruins. Feels no "sting," no
# confession-flare from touching it: to him it is just an inert historical
# relic, and he doesn't understand why anyone would carry "such a heavy,
# useless weight."
# gamma: a full generational/societal timeskip -- confirms the Hum's effects
# are genuinely PERMANENT, not just ongoing. This is strong RETROACTIVE
# CONFIRMING EVIDENCE for Ruling 6.1 (H elevated to floor status because the
# premise permanently dissolves structural bounds) -- the epilogue is the
# clearest possible validation of that original call.
eng.report('ch14alt.epilogue_child_finds_badge_generational_timeskip')

# Final image: the child sees a distant, solitary figure walking a rusted
# highway -- almost certainly Valeria. "The air around her seemed to shimmer...
# as if the truth was trying to find a way to become a lie again." Final line:
# "in the absolute, beautiful, agonizing silence of the new world, that was
# the only hope left."
eng.report('ch14alt.figure_on_the_horizon_FINAL_BEAT')

print()
print('══════════════════════════════════════════════════════════════')
print('G CANDIDACY AT THE MANUSCRIPT\'S ACTUAL ENDING -- NOT FIRED')
print('══════════════════════════════════════════════════════════════')
print('This is the literal last page of a complete, 14-chapter-plus-epilogue')
print('manuscript -- if G (Narrative Coherence: "all threads resolve, arcs land,')
print('thematic intent fulfilled") were ever going to fire, it would be here.')
print()
print('Declining to fire it, on the actual text:')
print("  - Thorne's ultimate fate/moral reckoning: never resolved. Last seen alive,")
print('    defeated, sitting in the wreckage (Ch.13) -- never mentioned again.')
print('  - The Vow/Silent Colonies social order: explicitly ONGOING and adapting,')
print('    not landed -- "a civilization of the Wary... learning to live in the')
print('    Light... building new masks, new distances." A process, not a resolution.')
print("  - The final image is DELIBERATELY ambiguous, not a clean landing: \"the")
print('    truth was trying to find a way to become a lie again" -- and the text')
print('    itself calls that ambiguity "the only hope left," not a resolved thesis.')
print()
print('NEW OPEN QUESTION FOR THE FRAMEWORK (not a manuscript defect -- a genuine')
print('gap in node coverage): neither F nor G describes a story that is fully')
print('FINISHED but deliberately ends on irresolution/ambiguity as its own')
print('narrative choice, rather than either "approaching" or "reaching" coherence.')
print('The current 8-node vocabulary may need a 9th node, or an explicit ruling on')
print('how to trace a thematically-open ending -- logged as an open question,')
print('not resolved here.')
print()

eng.summary()
print(f'\nFULL STRESS TEST (Ch.7-14+Epilogue): physics_violations={len(eng.physics_violations)}, '
      f'lore_creep={len(eng.violations)}, deferred_z_open={len(eng.deferred_z)}')


── Alt-Ch.13: The Anatomy of the wake (backfill + remainder) ──
(continues from ch13alt.thorne_defeated_they_leave, already traced above --
 backfilling two beats that chronologically precede it, for completeness)

  ch13alt.reboot_hum_gone_LOCALLY                    active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch13alt.cassie_ledger_stayed                       active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  NOTE: ch13alt.thorne_defeated_they_leave already traced (A(Valeria+Thorne)
    deactivated). Continuing forward from there.

  ch13alt.climb_to_surface                           active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch13alt.plaza_total_exposure_survivors             active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch13alt.valeria_becomes_skinless                   active=['H[floor]', 'A(Valeria)', 'A(Cassie+Valeria_tilted)', 'D']  slots=2
  ch13alt.ba